# 14y Verify VeRi and CityFlow ReID Checkpoints
GPU verifier for single-camera ReID checkpoint metrics: CityFlowV2 TransReID mAP, CLIP-SENet v6 VeRi-776 base and rerank metrics, and 09v TransReID VeRi-776 SIE metrics.

In [ ]:
from pathlib import Path
import json
import os
import platform
import shutil
import subprocess
import sys

REPO_URL = "https://github.com/MRKDaGods/gp.git"
EXPECTED_MASTER_SHA_AT_BUILD = "5ce6eb516896af17ec1b975b58fd15df89a2291e"
WORK_DIR = Path("/kaggle/working")
PROJECT = WORK_DIR / "gp"
RESULTS_PATH = WORK_DIR / "14y_verify_results.json"

def run(cmd, *, cwd=None, check=True, capture=False):
    print("$", " ".join(map(str, cmd)))
    if capture:
        return subprocess.check_output(list(map(str, cmd)), cwd=cwd, text=True)
    return subprocess.run(list(map(str, cmd)), cwd=cwd, check=check)

def pip_install(*args):
    run([sys.executable, "-m", "pip", "install", "-q", *args])

if not PROJECT.exists():
    run(["git", "clone", "--branch", "master", "--depth", "1", REPO_URL, str(PROJECT)])
else:
    run(["git", "-C", PROJECT, "fetch", "origin", "master", "--depth", "1"])
    run(["git", "-C", PROJECT, "checkout", "master"])
    run(["git", "-C", PROJECT, "reset", "--hard", "origin/master"])

os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))
head_sha = run(["git", "rev-parse", "HEAD"], capture=True).strip()
print("resolved git SHA:", head_sha)
print("expected master SHA at notebook build:", EXPECTED_MASTER_SHA_AT_BUILD)
assert head_sha == EXPECTED_MASTER_SHA_AT_BUILD, f"master moved: expected {EXPECTED_MASTER_SHA_AT_BUILD}, got {head_sha}"
print("python:", sys.version)
print("platform:", platform.platform())

pip_install("faiss-cpu", "motmetrics", "loguru", "omegaconf", "rich", "networkx>=3.1", "click")
pip_install("filterpy", "ftfy", "lapx", "timm")
pip_install("--no-deps", "ultralytics")
pip_install("--no-deps", "boxmot==11.0.3")
pip_install("--no-deps", "-e", ".")

help_text = run([sys.executable, "scripts/run_pipeline.py", "--help"], capture=True)
RUN_ID_SHIM_APPLIED = False
if "--run-id" not in help_text:
    script_path = Path("scripts/run_pipeline.py")
    script_text = script_path.read_text(encoding="utf-8")
    script_text = script_text.replace(
        '@click.option("--dry-run", is_flag=True, default=False, help="Print resolved plan without running stages")\n',
        '@click.option("--dry-run", is_flag=True, default=False, help="Print resolved plan without running stages")\n'
        '@click.option("--run-id", default=None, help="Run id/name for the output directory")\n',
    )
    script_text = script_text.replace(
        'def main(config: str, dataset_config: str, stages: str, smoke_test: bool, dry_run: bool, override: tuple):',
        'def main(config: str, dataset_config: str, stages: str, smoke_test: bool, dry_run: bool, run_id: str | None, override: tuple):',
    )
    script_text = script_text.replace(
        '    if apply_cpu_when_no_cuda(cfg):\n',
        '    if run_id:\n        cfg.project.run_name = run_id\n\n    if apply_cpu_when_no_cuda(cfg):\n',
    )
    script_path.write_text(script_text, encoding="utf-8")
    RUN_ID_SHIM_APPLIED = True
    print("Applied runtime-only --run-id compatibility shim")
else:
    print("run_pipeline.py already exposes --run-id")

for mount in [Path("/tmp"), WORK_DIR]:
    total, used, free = shutil.disk_usage(mount)
    print(f"{mount}: {free / 1024**3:.1f} GiB free / {total / 1024**3:.1f} GiB total")

pip_install("scikit-learn", "scipy", "pandas", "opencv-python-headless", "tqdm", "timm==1.0.11")

In [ ]:
from pathlib import Path
import hashlib
import json
import math
import os
import re
import shutil
import subprocess
import tarfile
import time

INPUT_ROOT = Path("/kaggle/input")

def write_json(path: Path, payload: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=2, sort_keys=True), encoding="utf-8")
    print(f"wrote {path}")

def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as fh:
        for chunk in iter(lambda: fh.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def symlink_or_copy(src: Path, dst: Path) -> None:
    if dst.exists() or dst.is_symlink():
        if dst.is_dir() and not dst.is_symlink():
            shutil.rmtree(dst)
        else:
            dst.unlink()
    dst.parent.mkdir(parents=True, exist_ok=True)
    try:
        dst.symlink_to(src, target_is_directory=src.is_dir())
    except OSError:
        if src.is_dir():
            shutil.copytree(src, dst)
        else:
            shutil.copy2(src, dst)

def list_input_roots(max_items: int = 80) -> list[str]:
    roots = sorted(str(path) for path in INPUT_ROOT.iterdir()) if INPUT_ROOT.exists() else []
    for item in roots[:max_items]:
        print(item)
    if len(roots) > max_items:
        print(f"... {len(roots) - max_items} more")
    return roots

def find_input_dir(owner_slug: str, *hints: str) -> Path | None:
    owner, slug = owner_slug.split("/", 1)
    direct_candidates = [
        INPUT_ROOT / slug,
        INPUT_ROOT / owner_slug.replace("/", "-"),
        INPUT_ROOT / "notebooks" / owner / slug,
        INPUT_ROOT / "datasets" / owner / slug,
    ]
    for candidate in direct_candidates:
        if candidate.exists():
            return candidate
    wanted = [slug.lower(), *(hint.lower() for hint in hints)]
    for path in INPUT_ROOT.rglob("*") if INPUT_ROOT.exists() else []:
        if path.is_dir():
            text = str(path).lower()
            if all(token in text for token in wanted if token):
                return path
    return None

def find_file(names: list[str], *, owner_slug: str | None = None, hints: tuple[str, ...] = ()) -> Path:
    roots = []
    if owner_slug:
        root = find_input_dir(owner_slug, *hints)
        if root is not None:
            roots.append(root)
    roots.append(INPUT_ROOT)
    matches = []
    lowered_hints = tuple(h.lower() for h in hints)
    for root in roots:
        if root is None or not root.exists():
            continue
        for name in names:
            matches.extend(root.rglob(name))
    if owner_slug or hints:
        filtered = []
        for path in matches:
            text = str(path).lower()
            if owner_slug and owner_slug.split("/", 1)[1].lower() in text:
                filtered.append(path)
            elif lowered_hints and all(h in text for h in lowered_hints):
                filtered.append(path)
        matches = filtered or matches
    matches = sorted(set(matches), key=lambda p: (len(str(p)), str(p)))
    if not matches:
        raise FileNotFoundError(f"Could not find {names} owner_slug={owner_slug} hints={hints}")
    print("selected", matches[0])
    return matches[0]

def find_wildtrack_root() -> Path:
    markers = {"Image_subsets", "annotations_positions", "calibrations"}
    for path in INPUT_ROOT.rglob("*") if INPUT_ROOT.exists() else []:
        if path.is_dir():
            children = {child.name for child in path.iterdir() if child.is_dir()}
            if markers.issubset(children):
                print("WILDTRACK root:", path)
                return path
    raise FileNotFoundError("Could not find WILDTRACK root with Image_subsets, annotations_positions, calibrations")

def find_cityflow_root() -> Path:
    candidates = [
        INPUT_ROOT / "data-aicity-2023-track-2",
        INPUT_ROOT / "datasets" / "thanhnguyenle" / "data-aicity-2023-track-2",
    ]
    for root in candidates:
        if root.exists():
            return root
    for path in INPUT_ROOT.rglob("*") if INPUT_ROOT.exists() else []:
        if path.is_dir() and ((path / "train").exists() or (path / "AIC22_Track1_MTMC_Tracking").exists()):
            return path
    raise FileNotFoundError("Could not find CityFlowV2/AIC22 dataset root")

def prepare_cityflow_layout(source_root: Path) -> Path:
    raw_parent = PROJECT / "data" / "raw"
    tmp_parent = Path("/tmp/datasets")
    tmp_parent.mkdir(parents=True, exist_ok=True)
    if raw_parent.exists() or raw_parent.is_symlink():
        if raw_parent.is_symlink() or raw_parent.is_file():
            raw_parent.unlink()
        else:
            shutil.rmtree(raw_parent)
    raw_parent.parent.mkdir(parents=True, exist_ok=True)
    raw_parent.symlink_to(tmp_parent, target_is_directory=True)
    data_raw = tmp_parent / "cityflowv2"
    data_raw.mkdir(parents=True, exist_ok=True)
    roots = []
    if (source_root / "AIC22_Track1_MTMC_Tracking").exists():
        roots.append(source_root / "AIC22_Track1_MTMC_Tracking")
    roots.append(source_root)
    for root in roots:
        for split in ["train", "validation", "test"]:
            split_dir = root / split
            if not split_dir.exists():
                continue
            for scene_dir in sorted(split_dir.iterdir()):
                if not scene_dir.is_dir():
                    continue
                for cam_dir in sorted(scene_dir.iterdir()):
                    if not cam_dir.is_dir():
                        continue
                    flat = data_raw / f"{scene_dir.name}_{cam_dir.name}"
                    if not flat.exists():
                        symlink_or_copy(cam_dir, flat)
    expected = ["S01_c001", "S01_c002", "S01_c003", "S02_c006", "S02_c007", "S02_c008"]
    missing = [cam for cam in expected if not (data_raw / cam / "gt" / "gt.txt").exists()]
    assert not missing, f"CityFlow GT camera layout missing: {missing} under {data_raw}"
    print("CityFlow layout ready:", data_raw)
    return data_raw

In [ ]:
import torch
print("cuda available:", torch.cuda.is_available())
assert torch.cuda.is_available(), "14y requires GPU (T4/P100 acceptable)"
DEVICE = "cuda:0"
list_input_roots()
CITYFLOW_ROOT = find_cityflow_root()
VERI_ROOT = None
for path in INPUT_ROOT.rglob("*") if INPUT_ROOT.exists() else []:
    if path.is_dir() and (path / "image_query").exists() and (path / "image_test").exists():
        VERI_ROOT = path
        break
assert VERI_ROOT is not None, "VeRi-776 root with image_query/image_test not found"
print("CityFlow root:", CITYFLOW_ROOT)
print("VeRi root:", VERI_ROOT)
Path("models/reid").mkdir(parents=True, exist_ok=True)
CF_CKPT = find_file(["transreid_cityflowv2_best.pth"], hints=("mtmc", "weights", "reid"))
VERI_CKPT = find_file(["vehicle_transreid_vit_base_veri776.pth"], hints=("mtmc", "weights", "reid"))
CLIPSENET_CKPT = find_file(["clipsenet_v6_veri776_best.pth", "vehicle_clip_senet_veri776.pth", "best.pth"], hints=("13", "clip", "senet"))
symlink_or_copy(CF_CKPT, PROJECT / "models" / "reid" / "transreid_cityflowv2_best.pth")
symlink_or_copy(VERI_CKPT, PROJECT / "models" / "reid" / "vehicle_transreid_vit_base_veri776.pth")
symlink_or_copy(CLIPSENET_CKPT, PROJECT / "models" / "reid" / "clipsenet_v6_veri776_best.pth")

In [ ]:
import re

metrics = []

def add_metric(model, dataset, checkpoint, metric, observed, target, tolerance, mode, required=True):
    row = {
        "model": model,
        "dataset": dataset,
        "checkpoint": checkpoint,
        "metric": metric,
        "observed": None if observed is None else float(observed),
        "target": target,
        "tolerance": tolerance,
        "mode": mode,
        "required": required,
        "passed": (not required) if observed is None else abs(float(observed) - target) <= tolerance,
    }
    metrics.append(row)
    return row

def parse_map_r1(text: str) -> tuple[float | None, float | None]:
    map_match = re.search(r"mAP\s*[:=]\s*([0-9.]+)%?", text, re.IGNORECASE)
    r1_match = re.search(r"(?:Rank-?1|R1)\s*[:=]\s*([0-9.]+)%?", text, re.IGNORECASE)
    def norm(match):
        if not match:
            return None
        value = float(match.group(1))
        return value / 100.0 if value > 1.0 else value
    return norm(map_match), norm(r1_match)

cmd = [
    sys.executable, "scripts/eval_cityflowv2_reid.py",
    "--weights", str(PROJECT / "models/reid/transreid_cityflowv2_best.pth"),
    "--data_root", str(CITYFLOW_ROOT),
    "--crop_dir", "/kaggle/working/crops/cityflowv2_reid_14y",
    "--img_size", "256", "256",
    "--batch_size", "128",
    "--device", DEVICE,
]
proc = subprocess.run(cmd, text=True, capture_output=True)
print(proc.stdout)
print(proc.stderr)
if proc.returncode != 0:
    raise RuntimeError("CityFlowV2 TransReID eval failed")
cf_map, cf_r1 = parse_map_r1(proc.stdout + "\n" + proc.stderr)
add_metric("TransReID ViT-B/16 CLIP", "CityFlowV2", "transreid_cityflowv2_best.pth", "map", cf_map, 0.8153, 0.005, "base", True)
add_metric("TransReID ViT-B/16 CLIP", "CityFlowV2", "transreid_cityflowv2_best.pth", "rank1", cf_r1, 0.9241, 0.005, "informational", False)

In [ ]:
# The CLIP-SENet v6 and 09v TransReID evals are notebook-first in this repo.
# This verifier records the exact intended contracts and imports the canonical notebooks as source artifacts.
# The run still fails if any required row is left unresolved, so a Kaggle PASS cannot hide a missing eval path.
NOTEBOOK_SOURCES = {
    "clipsenet_v6": "notebooks/kaggle/13e_clip_senet_eval/13e_clip_senet_eval.ipynb",
    "veri_09v": "notebooks/kaggle/09v_veri776_eval/09v-veri776-eval.ipynb",
}
for name, rel in NOTEBOOK_SOURCES.items():
    path = PROJECT / rel
    assert path.exists(), f"canonical source notebook missing: {rel}"
    print(name, path, path.stat().st_size)

unresolved = [
    ("CLIP-SENet v6", "VeRi-776", "clipsenet_v6_veri776_best.pth", "base_map", 0.8234, "base"),
    ("CLIP-SENet v6", "VeRi-776", "clipsenet_v6_veri776_best.pth", "base_rank1", 0.9654, "base"),
    ("CLIP-SENet v6", "VeRi-776", "clipsenet_v6_veri776_best.pth", "rerank_aqe_map", 0.9154, "rerank+AQE"),
    ("CLIP-SENet v6", "VeRi-776", "clipsenet_v6_veri776_best.pth", "rerank_aqe_rank1", 0.9732, "rerank+AQE"),
    ("09v TransReID v17", "VeRi-776", "vehicle_transreid_vit_base_veri776.pth", "best_rank1", 0.9833, "SIE 224x224"),
    ("09v TransReID v17", "VeRi-776", "vehicle_transreid_vit_base_veri776.pth", "best_map", 0.8997, "SIE 224x224"),
    ("09v TransReID v17", "VeRi-776", "vehicle_transreid_vit_base_veri776.pth", "joint_rank1", 0.9815, "SIE 224x224"),
    ("09v TransReID v17", "VeRi-776", "vehicle_transreid_vit_base_veri776.pth", "joint_map", 0.8971, "SIE 224x224"),
]
for model, dataset, checkpoint, metric, target, mode in unresolved:
    add_metric(model, dataset, checkpoint, metric, None, target, 0.005, mode, True)
add_metric("09v TransReID v17", "VeRi-776", "vehicle_transreid_vit_base_veri776.pth", "historical_r5_guard", 0.984505, 0.984505, 0.000001, "AQE k=3 + rerank, informational R5 not R1", False)

result = {
    "verifier": "14y_verify_veri_reid_checkpoints",
    "git_sha": head_sha,
    "run_id_shim_applied": RUN_ID_SHIM_APPLIED,
    "device": DEVICE,
    "datasets": {"cityflowv2_root": str(CITYFLOW_ROOT), "veri776_root": str(VERI_ROOT)},
    "checkpoints": {"cityflow_transreid": str(CF_CKPT), "clipsenet_v6": str(CLIPSENET_CKPT), "veri_transreid_09v": str(VERI_CKPT)},
    "metrics": metrics,
    "passed": all(row["passed"] for row in metrics if row.get("required", True)),
}
write_json(RESULTS_PATH, result)
print(json.dumps(result, indent=2))
failing = [row for row in metrics if row.get("required", True) and not row["passed"]]
assert not failing, "Required 14y notebook-first eval rows need executable extraction or inline implementation before this verifier can pass: " + json.dumps(failing, indent=2)